# 01 · Análisis de commits por dev

Productivity Tracker · Innovación Automotriz SpA

Analiza la actividad de commits del equipo en los repositorios de la organización `Auto-Innovation-Lab`, comparando el periodo antes/después de la adopción de Claude Code.

In [ ]:
import os
import requests
import pandas as pd
import matplotlib.pyplot as plt
from dotenv import load_dotenv

load_dotenv()  # lee .env en la raíz del repo (no versionado)

GITHUB_TOKEN = os.environ.get("GITHUB_TOKEN")
GITHUB_ORG = os.environ.get("GITHUB_ORG", "Auto-Innovation-Lab")
GITHUB_USERNAME = os.environ.get("GITHUB_USERNAME", "edesiosantos")

## Configuración

In [ ]:
from datetime import datetime

session = requests.Session()
session.headers.update({
    "Accept": "application/vnd.github+json",
    "X-GitHub-Api-Version": "2022-11-28",
})
if GITHUB_TOKEN:
    session.headers["Authorization"] = f"Bearer {GITHUB_TOKEN}"

# Fecha en la que el equipo empezó a usar Claude Code — ajustar según corresponda
AI_ADOPTION_DATE = datetime(2026, 1, 1)

# GitHub login -> datos del dev
TEAM = {
    "Ecxpectro": {"nombre": "Henrique Schraiber", "rol": "Agentic Engineer", "ia": "Claude Code"},
    "GuilhermeKill": {"nombre": "Guilherme Reis", "rol": "Agentic QA Engineer", "ia": "Claude Code"},
    "javelasquezb": {"nombre": "Javier Velásquez", "rol": "Tech Lead", "ia": "GPT"},
}

## Obtención de repositorios de la organización

In [ ]:
def get_org_repos(org, session):
    repos = []
    page = 1
    while True:
        resp = session.get(
            f"https://api.github.com/orgs/{org}/repos",
            params={"per_page": 100, "page": page, "type": "all"},
        )
        resp.raise_for_status()
        batch = resp.json()
        if not batch:
            break
        repos.extend(batch)
        page += 1
    return repos

repos = get_org_repos(GITHUB_ORG, session)
repo_names = [r["name"] for r in repos]
print(f"{len(repo_names)} repos encontrados en {GITHUB_ORG}:")
repo_names

## Extracción de commits por dev

In [ ]:
def get_repo_commits(org, repo, session):
    commits = []
    page = 1
    while True:
        resp = session.get(
            f"https://api.github.com/repos/{org}/{repo}/commits",
            params={"per_page": 100, "page": page},
        )
        if resp.status_code == 409:  # repo vacío, sin commits
            break
        resp.raise_for_status()
        batch = resp.json()
        if not batch:
            break
        commits.extend(batch)
        page += 1
    return commits

rows = []
for repo in repo_names:
    try:
        commits = get_repo_commits(GITHUB_ORG, repo, session)
    except requests.HTTPError as e:
        print(f"  ! error en {repo}: {e}")
        continue
    for c in commits:
        author = c.get("author")
        login = author["login"] if author else None
        rows.append({
            "repo": repo,
            "sha": c["sha"][:7],
            "author_login": login,
            "author_name": c["commit"]["author"]["name"],
            "date": pd.to_datetime(c["commit"]["author"]["date"]).tz_localize(None),
            "message": c["commit"]["message"].splitlines()[0],
        })
    print(f"{repo}: {len(commits)} commits")

commits_df = pd.DataFrame(rows)
commits_df["dev"] = commits_df["author_login"].map(
    lambda l: TEAM.get(l, {}).get("nombre", l)
).fillna(commits_df["author_name"])
commits_df["periodo"] = commits_df["date"].apply(
    lambda d: "Con IA" if d >= AI_ADOPTION_DATE else "Sin IA"
)
commits_df.head()

## Análisis y visualización

In [ ]:
# Commits totales por dev y periodo (antes/después de adopción IA)
resumen = commits_df.groupby(["dev", "periodo"]).size().unstack(fill_value=0)
display(resumen)

# Commits por semana por dev
commits_df["semana"] = commits_df["date"].dt.to_period("W").apply(lambda p: p.start_time)
semanal = commits_df.groupby(["semana", "dev"]).size().unstack(fill_value=0)

fig, ax = plt.subplots(figsize=(12, 5))
semanal.plot(ax=ax, marker="o")
ax.axvline(AI_ADOPTION_DATE, color="gray", linestyle="--", label="Adopción IA")
ax.set_title("Commits por semana por desarrollador")
ax.set_xlabel("Semana")
ax.set_ylabel("Commits")
ax.legend()
plt.tight_layout()
plt.show()